# project_lithos: Landslide4Sense Training on Kaggle

This notebook trains a U-Net model for landslide segmentation using the Kaggle dataset:

https://www.kaggle.com/datasets/tekbahadurkshetri/landslide4sense/data

Expected task: predict a landslide mask from remote sensing image data.

## 1. Imports and Setup

In [ ]:
import os
from pathlib import Path
import random
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Dataset Path

After adding the Kaggle dataset to this notebook, the files should appear under `/kaggle/input/landslide4sense/`.

In [ ]:
DATASET_ROOT = Path("/kaggle/input/datasets/tekbahadurkshetri/landslide4sense")
OUTPUT_DIR = Path("/kaggle/working/project_lithos_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset root exists:", DATASET_ROOT.exists())
print("Top-level dataset items:")
for item in sorted(DATASET_ROOT.iterdir()):
    print(" -", item)

## 3. Auto-Detect Landslide4Sense Folders

In [ ]:
def find_dir(root, candidates):
    root = Path(root)
    dirs = [p for p in root.rglob("*") if p.is_dir()]
    # Prefer exact relative suffix matches, in the order supplied.
    for candidate in candidates:
        suffix = candidate.lower().replace("\\", "/").strip("/")
        for path in dirs:
            rel = path.relative_to(root).as_posix().lower()
            if rel.endswith(suffix):
                return path
    return None

TRAIN_IMAGE_DIR = find_dir(DATASET_ROOT, [
    "TrainData/img", "TrainData/image", "TrainData/images",
    "train/img", "train/image", "train/images",
    "images/train", "img/train"
])

TRAIN_MASK_DIR = find_dir(DATASET_ROOT, [
    "TrainData/mask", "TrainData/masks", "TrainData/label", "TrainData/labels",
    "train/mask", "train/masks", "train/label", "train/labels",
    "masks/train", "labels/train"
])

VAL_IMAGE_DIR = find_dir(DATASET_ROOT, [
    "ValidData/img", "ValidData/image", "ValidData/images",
    "ValidationData/img", "ValidationData/image", "ValidationData/images",
    "valid/img", "valid/images", "val/img", "val/images"
])

VAL_MASK_DIR = find_dir(DATASET_ROOT, [
    "ValidData/mask", "ValidData/masks", "ValidData/label", "ValidData/labels",
    "ValidationData/mask", "ValidationData/masks",
    "ValidationData/label", "ValidationData/labels",
    "valid/mask", "valid/masks", "valid/label", "valid/labels",
    "val/mask", "val/masks", "val/label", "val/labels"
])

print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("TRAIN_MASK_DIR :", TRAIN_MASK_DIR)
print("VAL_IMAGE_DIR  :", VAL_IMAGE_DIR)
print("VAL_MASK_DIR   :", VAL_MASK_DIR)

if TRAIN_IMAGE_DIR is None or TRAIN_MASK_DIR is None:
    print("\\nDataset directory tree (directories only):")
    for path in sorted(p for p in DATASET_ROOT.rglob("*") if p.is_dir()):
        print(" -", path)
    raise FileNotFoundError(
        "Could not detect both training image and mask directories. "
        "Check the directory tree printed above."
    )


## 4. Dataset Reader

Landslide4Sense commonly stores samples as `.h5` files. This reader uses the image key `img` and mask keys such as `mask`, `label`, or `data` when available.

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
THRESHOLD = 0.5
IN_CHANNELS = 3

VALID_EXTENSIONS = {".h5", ".hdf5", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def first_dataset(group):
    for value in group.values():
        if isinstance(value, h5py.Dataset):
            return value[()]
        if isinstance(value, h5py.Group):
            nested = first_dataset(value)
            if nested is not None:
                return nested
    return None

def read_h5(path, preferred_keys):
    with h5py.File(path, "r") as file:
        for key in preferred_keys:
            if key in file and isinstance(file[key], h5py.Dataset):
                return file[key][()]
        data = first_dataset(file)
        if data is None:
            raise ValueError(f"No readable dataset found in {path}")
        return data

def resize_nearest(array, size):
    import cv2
    return cv2.resize(array, (size, size), interpolation=cv2.INTER_NEAREST)

def resize_linear(array, size):
    import cv2
    return cv2.resize(array, (size, size), interpolation=cv2.INTER_LINEAR)

def normalize_image(image):
    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 2:
        image = image[:, :, None]
    normalized = np.zeros_like(image, dtype=np.float32)
    for channel in range(image.shape[-1]):
        values = image[:, :, channel]
        finite = np.isfinite(values)
        if not finite.any():
            continue
        min_value = values[finite].min()
        max_value = values[finite].max()
        if max_value > min_value:
            normalized[:, :, channel] = np.where(
                finite, (values - min_value) / (max_value - min_value), 0.0
            )
    return np.nan_to_num(normalized, nan=0.0, posinf=0.0, neginf=0.0)

def sample_key(path):
    """Match image_134.h5 with mask_134.h5, even when prefixes differ."""
    stem = Path(path).stem.lower()
    numbers = re.findall(r"\d+", stem)
    if numbers:
        return numbers[-1]
    return re.sub(r"^(image|images|img|mask|masks|label|labels)[_-]*", "", stem)

class Landslide4SenseDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_size=128, channels=3):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.image_size = image_size
        self.channels = channels

        image_paths = sorted(
            p for p in self.image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        )
        mask_paths = sorted(
            p for p in self.mask_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        )
        if not image_paths:
            raise ValueError(f"No image files found in {self.image_dir}")
        if not mask_paths:
            raise ValueError(f"No mask files found in {self.mask_dir}")

        masks_by_key = {}
        for path in mask_paths:
            key = sample_key(path)
            if key in masks_by_key:
                raise ValueError(
                    f"Multiple mask files map to sample key {key!r}: "
                    f"{masks_by_key[key].name}, {path.name}. "
                    "Please inspect the dataset naming convention."
                )
            masks_by_key[key] = path

        self.samples = []
        missing = []
        for image_path in image_paths:
            key = sample_key(image_path)
            mask_path = masks_by_key.get(key)
            if mask_path is None:
                missing.append(image_path.name)
            else:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise ValueError(
                f"No image/mask pairs matched in {self.image_dir} and {self.mask_dir}. "
                f"Example image names: {[p.name for p in image_paths[:5]]}; "
                f"example mask names: {[p.name for p in mask_paths[:5]]}. "
                "Update sample_key() to match the dataset's naming convention."
            )

        print(
            f"Matched {len(self.samples)} image/mask pairs "
            f"({len(missing)} unmatched images skipped)."
        )
        if missing:
            print("Examples of unmatched images:", missing[:5])

    def __len__(self):
        return len(self.samples)

    def read_image(self, path):
        if path.suffix.lower() in {".h5", ".hdf5"}:
            image = np.asarray(read_h5(path, ["img", "image", "images", "data"]))
            if image.ndim == 2:
                image = image[:, :, None]
            # Convert channels-first HWC expectation to channels-last if needed.
            if image.ndim == 3 and image.shape[0] <= 20 and image.shape[-1] > 20:
                image = np.moveaxis(image, 0, -1)
            if image.ndim != 3:
                raise ValueError(f"Unexpected image shape {image.shape} in {path}")
            image = image[:, :, :self.channels]
            if image.shape[-1] < self.channels:
                missing_channels = self.channels - image.shape[-1]
                image = np.concatenate(
                    [image, np.repeat(image[:, :, -1:], missing_channels, axis=-1)],
                    axis=-1
                )
            image = resize_linear(image, self.image_size)
            return normalize_image(image)

        import cv2
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Could not read image file: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = resize_linear(image, self.image_size)
        return image.astype(np.float32) / 255.0

    def read_mask(self, path):
        if path.suffix.lower() in {".h5", ".hdf5"}:
            mask = np.asarray(read_h5(path, ["mask", "label", "labels", "y", "data"]))
        else:
            import cv2
            mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise ValueError(f"Could not read mask file: {path}")

        mask = np.squeeze(mask)
        if mask.ndim != 2:
            raise ValueError(f"Unexpected mask shape {mask.shape} in {path}")
        mask = resize_nearest(mask.astype(np.float32), self.image_size)
        # Landslide masks may be stored as 0/1 or 0/255.
        if mask.max(initial=0) > 1:
            mask = mask / 255.0
        return (mask > 0.5).astype(np.float32)

    def __getitem__(self, index):
        image_path, mask_path = self.samples[index]
        image = self.read_image(image_path)
        mask = self.read_mask(mask_path)
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        return image, mask

train_dataset = Landslide4SenseDataset(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, IMAGE_SIZE, IN_CHANNELS)

if VAL_IMAGE_DIR is not None and VAL_MASK_DIR is not None:
    val_dataset = Landslide4SenseDataset(VAL_IMAGE_DIR, VAL_MASK_DIR, IMAGE_SIZE, IN_CHANNELS)
else:
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

## 5. Visualize One Sample

In [ ]:
if len(train_dataset) == 0:
    raise RuntimeError("Training dataset contains no matched image/mask pairs.")

img, mask = train_dataset[0]
display_image = img[:3].permute(1, 2, 0).numpy()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(display_image)
plt.title("Input image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask.squeeze().numpy(), cmap="gray")
plt.title("Ground truth mask")
plt.axis("off")
plt.show()

print("Image tensor shape:", img.shape)
print("Mask tensor shape:", mask.shape)


## 6. U-Net Model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 32)
        self.down2 = DoubleConv(32, 64)
        self.down3 = DoubleConv(64, 128)
        self.down4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bridge = DoubleConv(256, 512)
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 32)
        self.output = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))
        bridge = self.bridge(self.pool(d4))
        x = self.up4(bridge)
        x = self.conv4(torch.cat([x, d4], dim=1))
        x = self.up3(x)
        x = self.conv3(torch.cat([x, d3], dim=1))
        x = self.up2(x)
        x = self.conv2(torch.cat([x, d2], dim=1))
        x = self.up1(x)
        x = self.conv1(torch.cat([x, d1], dim=1))
        return self.output(x)

model = UNet(in_channels=IN_CHANNELS).to(DEVICE)
print("Model created")

## 7. Metrics and Training Functions

In [ ]:
def dice_score(logits, masks, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * masks).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean()

def iou_score(logits, masks, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * masks).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3)) - intersection
    return ((intersection + eps) / (union + eps)).mean()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    
    progress = tqdm(loader, leave=False, desc="Train" if training else "Valid")
    for images, masks in progress:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, masks)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_score(logits.detach(), masks).item()
        total_iou += iou_score(logits.detach(), masks).item()
    
    count = len(loader)
    return total_loss / count, total_dice / count, total_iou / count

## 8. Train Model

In [ ]:
best_iou = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice, train_iou = run_epoch(train_loader, training=True)
    val_loss, val_dice, val_iou = run_epoch(val_loader, training=False)
    
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_dice": train_dice,
        "train_iou": train_iou,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
    })
    
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f}, train_dice={train_dice:.4f}, train_iou={train_iou:.4f} | "
        f"val_loss={val_loss:.4f}, val_dice={val_dice:.4f}, val_iou={val_iou:.4f}"
    )
    
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pt")
        print("Saved best model:", OUTPUT_DIR / "best_model.pt")

print("Best validation IoU:", best_iou)

## 9. Plot Training History

In [ ]:
epochs = [row["epoch"] for row in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, [row["train_loss"] for row in history], label="Train loss")
plt.plot(epochs, [row["val_loss"] for row in history], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(epochs, [row["train_iou"] for row in history], label="Train IoU")
plt.plot(epochs, [row["val_iou"] for row in history], label="Val IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()
plt.title("IoU")
plt.show()

## 10. Visualize Predictions

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pt", map_location=DEVICE))
model.eval()

images, masks = next(iter(val_loader))
images = images.to(DEVICE)

with torch.no_grad():
    logits = model(images)
    probs = torch.sigmoid(logits).cpu()
    preds = (probs > THRESHOLD).float()

images = images.cpu()

sample_count = min(4, images.shape[0])
plt.figure(figsize=(12, sample_count * 3))

for i in range(sample_count):
    rgb = images[i, :3].permute(1, 2, 0).numpy()
    true_mask = masks[i, 0].numpy()
    pred_mask = preds[i, 0].numpy()
    
    plt.subplot(sample_count, 3, i * 3 + 1)
    plt.imshow(rgb)
    plt.title("Input")
    plt.axis("off")
    
    plt.subplot(sample_count, 3, i * 3 + 2)
    plt.imshow(true_mask, cmap="gray")
    plt.title("Ground truth")
    plt.axis("off")
    
    plt.subplot(sample_count, 3, i * 3 + 3)
    plt.imshow(pred_mask, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 11. Save Final Model

The trained model is saved in Kaggle working output. Download `best_model.pt` from the notebook output files.

In [27]:
print("Best model path:", OUTPUT_DIR / "best_model.pt")
print("Kaggle output folder:", OUTPUT_DIR)
print("Done")

Best model path: /kaggle/working/project_lithos_outputs/best_model.pt
Kaggle output folder: /kaggle/working/project_lithos_outputs
Done


In [28]:
/kaggle/working/project_lithos_outputs/best_model.pt

NameError: name 'kaggle' is not defined